Анализ эффективности работы отдела продаж:
1. Оцените эффективность отдельных владельцев сделок
с точки зрения количества обработанных сделок, коэффициента конверсии и
общей суммы продаж.

In [3]:
import pandas as pd

deals_df = pd.read_excel("Deals_f_clean.xlsx", dtype={"Id": str,"Contact Name": str})

display(deals_df.head())

deals_df.info()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch,SLA_hours
0,5805028000056864695,Ben Hall,NaT,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,2024-06-21 15:30:00,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaT,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Morning,2024-06-21 15:23:00,6.0,NaN,0.0,2000.0,5805028000056834471,NaN,NaN,NaN
2,5805028000056832357,Ulysses Adams,2024-06-21,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,2024-06-21 14:45:00,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN,0.4453
3,5805028000056824246,Eva Kent,2024-06-21,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,2024-06-21 13:32:00,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN,1.0011
4,5805028000056873292,Ben Hall,2024-06-21,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,2024-06-21 13:21:00,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN,0.8867


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19822 entries, 0 to 19821
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Id                   19822 non-null  object        
 1   Deal Owner Name      19793 non-null  object        
 2   Closing Date         13136 non-null  datetime64[ns]
 3   Quality              17586 non-null  object        
 4   Stage                19822 non-null  object        
 5   Lost Reason          14353 non-null  object        
 6   Page                 19822 non-null  object        
 7   Campaign             15583 non-null  object        
 8   SLA                  14995 non-null  object        
 9   Content              13798 non-null  object        
 10  Term                 12101 non-null  object        
 11  Source               19822 non-null  object        
 12  Payment Type         483 non-null    object        
 13  Product              3537 non-n

In [6]:
# --- 1. Уберем строки без владельца ---
df = deals_df.dropna(subset=["Deal Owner Name"]).copy()

# --- 2. Считаем общие и успешные сделки ---
summary = (
    df.groupby("Deal Owner Name")
    .agg(
        total_deals=("Id", "count"),
        successful_deals=("Stage", lambda x: (x == "Payment Done").sum()),
        total_sales=("Offer Total Amount", lambda x: x[df.loc[x.index, "Stage"] == "Payment Done"].sum())
    )
    .reset_index()
)

# --- 3. Добавляем коэффициент конверсии и среднюю стоимость сделки ---
summary["conversion_rate (%)"] = (
    summary["successful_deals"] / summary["total_deals"] * 100
).round(2)

summary["avg_sale"] = (
    summary["total_sales"] / summary["successful_deals"]
).fillna(0).round(2)

# --- 4. Переставляем столбцы ---
summary = summary[
    ["Deal Owner Name", "total_deals", "successful_deals", "conversion_rate (%)", "avg_sale", "total_sales"]
]

# --- 5. Сортируем по сумме продаж ---
summary = summary.sort_values(by="total_sales", ascending=False).reset_index(drop=True)

# --- 6. Отображаем красиво ---
print("📊 Эффективность владельцев сделок:")
print(summary.to_string(index=False))

# --- 7. Сохраняем в JSON для Dash ---
summary.to_json("deal_owners_summary.json", orient="records", force_ascii=False, indent=4)


📊 Эффективность владельцев сделок:
Deal Owner Name  total_deals  successful_deals  conversion_rate (%)  avg_sale  total_sales
  Charlie Davis         2799               148                 5.29   7121.62    1054000.0
  Ulysses Adams         2069               141                 6.81   7145.39    1007500.0
   Julia Nelson         2085                92                 4.41   7711.97     709501.0
Paula Underwood         1771                93                 5.25   7451.61     693000.0
  Oliver Taylor          160                50                31.25  10490.00     524500.0
 Quincy Vincent         1805                65                 3.60   7153.85     465000.0
       Ben Hall         1303                46                 3.53   7489.13     344500.0
  Victor Barnes         1187                44                 3.71   7634.09     335900.0
     Nina Scott         1218                46                 3.78   6782.61     312000.0
     Jane Smith          904                31         

И для наглядности еще посмотрим на топ 5

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Топ-5 по конверсии и продажам
top_conv = summary.nlargest(5, "conversion_rate (%)")
top_sales = summary.nlargest(5, "total_sales")

# Создаем подграфики
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("🏆 Топ-5 по коэффициенту конверсии", "💰 Топ-5 по сумме продаж"),
    horizontal_spacing=0.15
)

# --- Левый график: конверсия ---
fig.add_trace(
    go.Bar(
        x=top_conv["Deal Owner Name"],
        y=top_conv["conversion_rate (%)"],
        text=top_conv["conversion_rate (%)"].round(1).astype(str) + "%",
        textposition="outside",
        marker_color="#2E91E5",  # единый цвет
        name="Конверсия"
    ),
    row=1, col=1
)

# --- Правый график: сумма продаж ---
fig.add_trace(
    go.Bar(
        x=top_sales["Deal Owner Name"],
        y=top_sales["total_sales"],
        text=top_sales["total_sales"].round(0).astype(int).astype(str),
        textposition="outside",
        marker_color="#00CC96",  # единый цвет, можно заменить
        name="Сумма продаж"
    ),
    row=1, col=2
)

# Настройка внешнего вида
fig.update_layout(
    height=500,
    width=950,
    showlegend=False,
    title_text="Эффективность менеджеров: конверсия и сумма продаж",
    title_x=0.5,
    title_font=dict(size=18),
)

fig.update_xaxes(title_text="Владелец сделки", tickangle=45)
fig.update_yaxes(title_text="Конверсия, %", row=1, col=1)
fig.update_yaxes(title_text="Сумма продаж", row=1, col=2)

fig.show()


In [9]:
fig.write_json("owners_efficiency_subplots.json")
